# Scraper Tempat Makan & Wisata Viral di Indonesia (6 Bulan Terakhir)

Notebook ini melakukan scraping berita dari **Google News RSS** (bahasa Indonesia) untuk mencari artikel tentang tempat makan/wisata yang sedang viral dalam 6 bulan terakhir, lalu mengekstrak:

- Nama tempat
- Lokasi (kelurahan, kecamatan, kota/kabupaten, provinsi) + latitude/longitude (via geocoding OpenStreetMap Nominatim)
- Sejak kapan viral (perkiraan dari tanggal berita/isi artikel)
- Kenapa viral (kalimat yang mengandung alasan viral)
- Sumber berita (judul, link, tanggal publish)

## Catatan penting
- Ekstraksi nama tempat, lokasi, dan alasan viral memakai **heuristik berbasis regex**, bukan NLP/NER penuh, karena berita berbahasa Indonesia sangat bervariasi formatnya. Hasil disarankan **dicek/dibersihkan manual** sebelum dipakai untuk keperluan resmi.
- Geocoding memakai **Nominatim (OpenStreetMap)** yang gratis tapi ada batas 1 request/detik dan mewajibkan `user_agent` yang jelas. Untuk kebutuhan volume besar/produksi, pertimbangkan pakai Google Maps Geocoding API (berbayar) untuk akurasi lebih baik.
- Gunakan data ini untuk riset/analisis internal. Selalu cek Terms of Service situs sumber sebelum melakukan scraping berskala besar.

In [1]:
import json
import re
import time
import urllib.parse
from datetime import datetime, timedelta, timezone

import feedparser
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from googlenewsdecoder import gnewsdecoder
from tqdm.auto import tqdm
import os

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

c:\Users\Agus Zainal Arifin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# KEYWORDS = [
#     "tempat makan viral",
#     "kuliner viral",
#     "cafe viral",
#     "restoran viral",
#     "wisata viral",
#     "destinasi wisata viral",
#     "spot wisata viral",
#     "tempat wisata baru viral",
# ]

# MONTHS_BACK = 6           # rentang waktu maksimal (bulan)
# MAX_ITEMS_PER_KEYWORD = 25  # batas jumlah artikel per keyword dari RSS
# REQUEST_DELAY = 1.0        # jeda antar request (detik) - sopan ke server

# CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=30 * MONTHS_BACK)
# print("Ambil artikel sejak:", CUTOFF_DATE.date())

In [3]:
BASE_KEYWORDS = [
    "tempat makan viral",
    "tempat makan enak",
    "kuliner viral",
    "kuliner enak",
    "cafe viral",
    "cafe enak",
    "restoran viral",
    "restoran enak",
    "wisata viral",
    "wisata baru"
]

PROVINCES = [
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Kepulauan Riau",
    "Jambi", "Sumatera Selatan", "Kepulauan Bangka Belitung", "Bengkulu", "Lampung",
    "DKI Jakarta", "Jawa Barat", "Banten", "Jawa Tengah", "DI Yogyakarta", "Jawa Timur",
    "Bali", "Nusa Tenggara Barat", "Nusa Tenggara Timur", "Kalimantan Barat",
    "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur", "Kalimantan Utara",
    "Sulawesi Utara", "Gorontalo", "Sulawesi Tengah", "Sulawesi Barat", "Sulawesi Selatan",
    "Sulawesi Tenggara", "Maluku", "Maluku Utara", "Papua", "Papua Barat",
    "Papua Selatan", "Papua Tengah", "Papua Pegunungan", "Papua Barat Daya",
]

BIG_CITIES = [
    "Jakarta Selatan", "Jakarta Pusat", "Jakarta Barat", "Jakarta Timur", "Jakarta Utara",
    "Surabaya", "Bandung", "Medan", "Yogyakarta", "Sleman", "Semarang", "Makassar",
    "Palembang", "Depok", "Bekasi", "Tangerang", "Bogor", "Malang", "Denpasar",
    "Bandar Lampung", "Padang", "Pekanbaru", "Banjarmasin", "Balikpapan", "Samarinda",
    "Pontianak", "Manado", "Jayapura", "Solo", "Cirebon", "Tasikmalaya", "Batam",
]

REGIONAL_CATEGORIES = [
    "tempat makan viral",
    "tempat makan enak",
    "cafe viral",
    "cafe enak",
    "restoran viral",
    "restoran enak",
    "wisata viral",
    "wisata baru",
    # "tempat belanja viral"
]

LOCATION_KEYWORDS = [
    f"{category} {region}"
    for region in PROVINCES + BIG_CITIES
    for category in REGIONAL_CATEGORIES
]
KEYWORDS = list(dict.fromkeys(BASE_KEYWORDS + LOCATION_KEYWORDS))

MONTHS_BACK = 6
MAX_ITEMS_PER_KEYWORD = 100
REQUEST_DELAY = 1.0

CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=30 * MONTHS_BACK)
print("Total keyword:", len(KEYWORDS))
print("Ambil artikel sejak:", CUTOFF_DATE.date())

Total keyword: 570
Ambil artikel sejak: 2026-04-05


In [19]:
EDITIONS = [
    ("id", "ID", "ID:id")
    # ("en-ID", "ID", "ID:en"),
]


def fetch_google_news(keyword: str, max_items: int = 25, hl: str = "id", gl: str = "ID", ceid: str = "ID:id"):
    """Ambil daftar berita dari Google News RSS untuk sebuah keyword & edisi bahasa/negara tertentu."""
    query = urllib.parse.quote(keyword)
    url = f"https://news.google.com/rss/search?q={query}&hl={hl}&gl={gl}&ceid={ceid}"
    feed = feedparser.parse(url)

    items = []
    for entry in feed.entries[:max_items]:
        try:
            pub_date = dateparser.parse(entry.published)
            if pub_date.tzinfo is None:
                pub_date = pub_date.replace(tzinfo=timezone.utc)
        except Exception:
            pub_date = None

        if pub_date is not None and pub_date < CUTOFF_DATE:
            continue  # lewati berita yang lebih lama dari batas waktu

        source = entry.get("source", {}).get("title") if hasattr(entry, "get") else None
        items.append(
            {
                "keyword": keyword,
                "title": entry.title,
                "link": entry.link,
                "published": pub_date,
                "source": source or getattr(getattr(entry, "source", None), "title", None),
            }
        )
    return items


all_news = []
os.makedirs("news", exist_ok=True)
for kw in tqdm(KEYWORDS, desc="Fetch RSS per keyword"):
    for hl, gl, ceid in EDITIONS:
        all_news.extend(fetch_google_news(kw, MAX_ITEMS_PER_KEYWORD, hl=hl, gl=gl, ceid=ceid))
        time.sleep(REQUEST_DELAY)

df_news = pd.DataFrame(all_news).drop_duplicates(subset="link").reset_index(drop=True)
# Simpan semua berita yang telah diambil ke dalam folder "news" tiap keyword
df_news.to_csv("news/all_news.csv", index=False)
for kw in KEYWORDS:
    df_news_kw = df_news[df_news["keyword"] == kw]
    df_news_kw.to_csv(f"news/news_{kw}.csv", index=False)
print(f"Total artikel ditemukan: {len(df_news)}")
df_news

Fetch RSS per keyword:   0%|          | 0/570 [00:00<?, ?it/s]

Fetch RSS per keyword: 100%|██████████| 570/570 [19:28<00:00,  2.05s/it]


Total artikel ditemukan: 5261


,keyword,title,link,published,source
0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang J...,https://news.google.com/rss/articles/CBMitAFBV...,2026-10-01 05:00:00+00:00,food.detik.com
1,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com
2,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Tak...",https://news.google.com/rss/articles/CBMizwFBV...,2026-09-16 07:00:00+00:00,IDN Times Jatim
3,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ad...",https://news.google.com/rss/articles/CBMidEFVX...,2026-09-27 10:22:00+00:00,Aktualita.co
4,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX...,2026-06-09 07:00:00+00:00,nibble.id
...,...,...,...,...,...
5256,wisata baru Batam,HARRIS Barelang Batam Hadirkan Promo Liburan K...,https://news.google.com/rss/articles/CBMiigFBV...,2026-07-07 07:00:00+00:00,Liputan Bhagasasi
5257,wisata baru Batam,"Pemko Batam Bangun Taman Budaya Rp50 Miliar, T...",https://news.google.com/rss/articles/CBMiyAFBV...,2026-07-15 07:00:00+00:00,Jawa Pos
5258,wisata baru Batam,Disbudpar: 113.784 wisman kunjungi Batam pada ...,https://news.google.com/rss/articles/CBMingFBV...,2026-06-03 07:00:00+00:00,ANTARA News Kepri
5259,wisata baru Batam,"Ascott Resmikan Fox Hotel Nagoya Batam, Hotel ...",https://news.google.com/rss/articles/CBMioAFBV...,2026-05-20 07:00:00+00:00,Beritakota


In [20]:
df_news.keyword.value_counts()

keyword
wisata baru                             99
kuliner viral                           94
wisata viral                            76
tempat makan viral                      73
wisata viral Bali                       68
                                        ..
restoran enak Kalimantan Utara           1
restoran viral Tasikmalaya               1
cafe viral Kepulauan Bangka Belitung     1
cafe enak Kepulauan Bangka Belitung      1
tempat makan viral Manado                1
Name: count, Length: 491, dtype: int64

In [4]:
df_news = pd.read_csv("news/all_news.csv")
df_news 

,keyword,title,link,published,source
0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang J...,https://news.google.com/rss/articles/CBMitAFBV...,2026-10-01 05:00:00+00:00,food.detik.com
1,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com
2,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Tak...",https://news.google.com/rss/articles/CBMizwFBV...,2026-09-16 07:00:00+00:00,IDN Times Jatim
3,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ad...",https://news.google.com/rss/articles/CBMidEFVX...,2026-09-27 10:22:00+00:00,Aktualita.co
4,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX...,2026-06-09 07:00:00+00:00,nibble.id
...,...,...,...,...,...
5256,wisata baru Batam,HARRIS Barelang Batam Hadirkan Promo Liburan K...,https://news.google.com/rss/articles/CBMiigFBV...,2026-07-07 07:00:00+00:00,Liputan Bhagasasi
5257,wisata baru Batam,"Pemko Batam Bangun Taman Budaya Rp50 Miliar, T...",https://news.google.com/rss/articles/CBMiyAFBV...,2026-07-15 07:00:00+00:00,Jawa Pos
5258,wisata baru Batam,Disbudpar: 113.784 wisman kunjungi Batam pada ...,https://news.google.com/rss/articles/CBMingFBV...,2026-06-03 07:00:00+00:00,ANTARA News Kepri
5259,wisata baru Batam,"Ascott Resmikan Fox Hotel Nagoya Batam, Hotel ...",https://news.google.com/rss/articles/CBMioAFBV...,2026-05-20 07:00:00+00:00,Beritakota


In [5]:
def resolve_and_fetch(url: str, timeout: tuple[int, int] = (5, 15)):
    """Decode URL Google News lalu ambil HTML halaman artikel sumber."""
    source_url = url
    if "news.google.com" in url:
        try:
            decoded = gnewsdecoder(url, interval=1, timeout=10)
            if decoded.get("success") and decoded.get("decoded_url"):
                source_url = decoded["decoded_url"]
        except Exception:
            pass

    try:
        resp = requests.get(source_url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        return resp.url, resp.text
    except requests.RequestException:
        return None, None


def extract_article_text(html: str) -> str:
    """Ambil isi artikel dari JSON-LD, meta description, paragraf, atau body HTML."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "lxml")

    # Banyak situs berita menyimpan isi artikel di JSON-LD, bukan tag p.
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            payload = json.loads(script.string or script.get_text())
        except (TypeError, json.JSONDecodeError):
            continue
        candidates = payload if isinstance(payload, list) else [payload]
        for item in candidates:
            if isinstance(item, dict) and isinstance(item.get("articleBody"), str):
                article_body = re.sub(r"\s+", " ", item["articleBody"]).strip()
                if len(article_body) >= 100:
                    return article_body

    meta = soup.find("meta", attrs={"name": "description"}) or soup.find(
        "meta", attrs={"property": "og:description"}
    )
    meta_text = meta.get("content", "").strip() if meta else ""

    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paragraphs = [p for p in paragraphs if len(p) >= 20]
    paragraph_text = " ".join(paragraphs)
    if len(paragraph_text) >= 100:
        return paragraph_text
    if len(meta_text) >= 40:
        return meta_text

    body_text = soup.get_text(" ", strip=True) if soup.body else ""
    return re.sub(r"\s+", " ", body_text).strip()


# def fetch_article_batch(
#     df_news: pd.DataFrame,
#     mulai: int = 0,
#     max_articles: int = 2410,
#     request_delay: float = REQUEST_DELAY,
# ) -> tuple[list[str | None], list[str]]:
#     """Fetch dan ekstrak isi artikel sebanyak maksimal `max_articles` baris."""
#     article_texts = []
#     final_urls = []

#     for article_number, link in enumerate(df_news["link"].head(max_articles), start=1):
#         if article_number < mulai: continue
#         print("Processing article number:", article_number)
#         print("Fetching link:", link)
#         final_url, html = resolve_and_fetch(link)
#         final_urls.append(final_url)
#         article_texts.append(extract_article_text(html))
#         print("panjang artikel:", len(article_texts[-1]))
#         time.sleep(request_delay)

#     return final_urls, article_texts


# final_urls, article_texts = fetch_article_batch(df_news, mulai=0, max_articles=2410)

In [6]:
import concurrent.futures

# Kenapa loop suka "diam" tanpa error: gnewsdecoder/requests kadang stall di level
# socket (koneksi macet, bukan timeout HTTP biasa) sehingga thread utama menggantung
# selamanya. fetch_article_safe menjalankan fetch di thread terpisah dan memaksa
# menyerah setelah HARD_TIMEOUT detik, supaya satu link bermasalah tidak mematikan
# seluruh loop.
HARD_TIMEOUT = 25.0
_fetch_executor = concurrent.futures.ThreadPoolExecutor(max_workers=4, thread_name_prefix="fetch")


def fetch_article_safe(link: str, hard_timeout: float = HARD_TIMEOUT):
    """resolve_and_fetch + extract_article_text dengan hard timeout & anti-exception."""
    future = _fetch_executor.submit(resolve_and_fetch, link)
    try:
        final_url, html = future.result(timeout=hard_timeout)
    except Exception:
        return None, ""
    try:
        return final_url, extract_article_text(html)
    except Exception:
        return final_url, ""

In [ ]:
import json
import os
from pathlib import Path

# Checkpoint per-baris: setiap link yang selesai difetch langsung ditulis ke file
# ini (flush + fsync). Kalau kernel macet/di-restart, jalankan ulang cell ini —
# link yang sudah ada di checkpoint otomatis dilewati (resume), jadi progres tidak
# hilang lagi walau berhenti di tengah jalan.
CHECKPOINT_PATH = Path("fetch_checkpoint.jsonl")

START_FROM = 0
done = {}
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                done[rec["index"]] = rec
            except json.JSONDecodeError:
                continue
    print(f"Resume: {len(done)} link sudah pernah diproses, akan dilewati.")

with CHECKPOINT_PATH.open("a", encoding="utf-8") as ckpt_file:
    for idx, link in enumerate(tqdm(df_news[START_FROM:]["link"], desc="Fetch isi artikel")):
        if idx + START_FROM in done:
            continue
        try:
            final_url, text = fetch_article_safe(link)
        except Exception as e:
            print(f"Error tak terduga pada index {idx}: {e}")
            final_url, text = None, ""

        rec = {"index": idx + START_FROM, "link": link, "final_url": final_url, "article_text": text}
        ckpt_file.write(json.dumps(rec, ensure_ascii=False) + "\n")
        ckpt_file.flush()
        os.fsync(ckpt_file.fileno())
        done[idx] = rec
        time.sleep(REQUEST_DELAY)

print(f"Total link diproses (checkpoint): {len(done)} dari {len(df_news)}")

Fetch isi artikel:  96%|█████████▌| 5061/5261 [4:57:08<11:36,  3.48s/it]   

In [ ]:
# Gabungkan seluruh checkpoint (boleh dijalankan kapan saja, termasuk saat proses
# fetch di atas belum 100% selesai, untuk lihat progres sementara) jadi satu CSV.
indices = sorted(i for i in done if i < len(df_news))
df_result = df_news.iloc[indices].copy()
df_result["final_url"] = [done[i]["final_url"] for i in indices]
df_result["article_text"] = pd.Series(
    [done[i]["article_text"] for i in indices], index=df_result.index, dtype="string"
)
df_result["article_text_length"] = df_result["article_text"].str.len().fillna(0).astype(int)

print(f"Artikel dengan isi berhasil diambil: {(df_result['article_text_length'] >= 100).sum()} dari {len(df_result)}")
print(f"Fetch gagal/kosong: {df_result['article_text'].eq('').sum()}")

df_result_final = df_result[df_result["article_text_length"] >= 100].reset_index(drop=True)
df_result_final.to_csv("df_news_extended_full_coba.csv", index=False)
print("Saved df_news_extended_full_coba.csv")

In [ ]:
import glob
import json
import os
import re
import time
from datetime import datetime, timedelta

import pandas as pd
from dateutil import parser as dateparser
from rapidfuzz import fuzz
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

In [5]:
NEWS_CSV_FILE = "df_news_extended_full_coba.csv"
MIN_ARTICLE_LEN = 150

df_news_all = pd.read_csv(NEWS_CSV_FILE)
print("File dipakai:", NEWS_CSV_FILE)

# beberapa batch mungkin punya nama kolom isi artikel yang sedikit berbeda; standardisasi
text_col_candidates = [c for c in ["article_text", "article_text_extended", "text"] if c in df_news_all.columns]
if "article_text" not in df_news_all.columns and text_col_candidates:
    df_news_all["article_text"] = df_news_all[text_col_candidates[0]]

dedup_key = "final_url" if "final_url" in df_news_all.columns else "link"
df_news_all = df_news_all.dropna(subset=[dedup_key])
df_news_all = df_news_all.drop_duplicates(subset=[dedup_key]).reset_index(drop=True)
df_news_all = df_news_all[df_news_all["article_text"].fillna("").str.len() >= MIN_ARTICLE_LEN].reset_index(drop=True)

print("Total artikel unik & layak diproses:", len(df_news_all))
df_news_all.head(3)

File dipakai: df_news_extended_full_coba.csv
Total artikel unik & layak diproses: 4834


,keyword,title,link,published,source,final_url,article_text,article_text_length
0,tempat makan viral,5 Ramen Lezat di Jakarta Ini Langganan Orang J...,https://news.google.com/rss/articles/CBMitAFBV...,2026-10-01 05:00:00+00:00,food.detik.com,https://food.detik.com/tempat-makan/d-8687381/...,Semangkuk ramen berkuah gurih hangat dengan an...,4249
1,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com,https://jatimnet.com/media-sosial-mengubah-car...,"Jumat, 25 September 2026 08:00 UTC Ilustrasi: ...",7110
2,tempat makan viral,"10 Rekomendasi Kuliner Malam Sidoarjo, Gak Tak...",https://news.google.com/rss/articles/CBMizwFBV...,2026-09-16 07:00:00+00:00,IDN Times Jatim,https://jatim.idntimes.com/food/dining-guide/1...,Kuliner malam di Sidoarjo menjadi daya tarik t...,7524


## 5. Buat Prompt Batch untuk Ekstraksi Manual via Chat Claude.ai

Sel di bawah membuat file prompt per-batch (`manual_llm_batches_coba/batch_XXX_prompt.txt`) yang bisa langsung ditempel ke chat Claude.ai untuk diekstrak jadi daftar tempat dengan skema kolom final:

`no, nama_tempat, kategori, jenis_tempat, lokasi, keterangan, status_viral, viral_sejak, dasar_viral, kelurahan_desa, kecamatan, kota_kabupaten, provinsi, negara, latitude, longitude, presisi_koordinat, catatan_lokasi, jumlah_artikel, keyword_pencarian, judul_artikel_sumber, link_sumber`

Langkahnya:
1. Jalankan sel "Generate Prompt Batch" — ini membuat file `batch_001_prompt.txt`, `batch_002_prompt.txt`, dst di folder `manual_llm_batches_coba/`.
2. Buka chat baru di claude.ai, tempel **seluruh isi** satu file `batch_XXX_prompt.txt`, kirim.
3. Simpan balasan JSON Claude (hanya JSON, tanpa teks tambahan) sebagai `manual_llm_batches_coba/batch_XXX_response.json`.
4. Ulangi untuk semua batch, lalu jalankan sel "Gabungkan Hasil ke CSV Final" di bawah untuk membangun tabel final sesuai skema kolom di atas.

Kolom `no`, `jumlah_artikel`, `keyword_pencarian`, `judul_artikel_sumber`, `link_sumber` **tidak** diminta dari Claude — `jumlah_artikel` dihitung otomatis saat tempat yang sama disebut di beberapa artikel (fuzzy dedup berdasarkan nama), sisanya diisi dari metadata artikel saat hasil digabungkan.

In [6]:
MANUAL_BATCH_DIR = "manual_llm_batches_coba"
MANUAL_BATCH_SIZE = 8       # jumlah artikel per file prompt - kecilkan jika artikel panjang
MANUAL_MAX_CHARS = 3000     # potong isi artikel supaya prompt tidak terlalu panjang untuk chat

os.makedirs(MANUAL_BATCH_DIR, exist_ok=True)

# simpan pemetaan article_id -> metadata artikel, dipakai lagi saat mengimpor hasil balasan
id_map = {}
for idx, row in df_news_all.iterrows():
    id_map[str(idx)] = {
        "final_url": row[dedup_key],
        "title": row.get("title", ""),
        "link": row.get("link", row[dedup_key]),
        "keyword": row.get("keyword", ""),
    }
with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), "w", encoding="utf-8") as f:
    json.dump(id_map, f, ensure_ascii=False, indent=2)

PROMPT_HEADER = """Kamu akan menerima beberapa artikel berita. Untuk SETIAP artikel, ekstrak semua tempat makan/kafe/resto/wisata/museum/wahana/pameran yang disebutkan.

Balas HANYA dengan JSON valid (tanpa penjelasan, tanpa markdown code fence) dengan format:
{
  "results": [
    {
      "article_id": <id artikel, angka>,
      "places": [
        {
          "nama_tempat": "...",
          "kategori": "Kafe/Restoran/Wisata Alam/Museum/Wahana/Pameran/lainnya",
          "jenis_tempat": "subtipe lebih spesifik, mis. 'rooftop cafe', 'kuliner kaki lima', 'air terjun', 'taman hiburan', dst",
          "lokasi": "lokasi mentah apa adanya sesuai artikel (alamat/nama jalan/area jika disebut)",
          "keterangan": "ringkasan singkat ciri khas/daya tarik tempat ini (mis. menu andalan, konsep, fasilitas)",
          "status_viral": "Disebut viral" (artikel eksplisit menyebut tempat ini viral/ramai dibahas/jadi perbincangan warganet) atau "Rekomendasi" (tempat hanya direkomendasikan/masuk daftar, tanpa disebut eksplisit viral),
          "viral_sejak": "tanggal/bulan/periode eksplisit terkait mulai viral jika disebut, atau 'Tidak disebut' jika tidak ada",
          "dasar_viral": "kutipan/alasan singkat dari artikel yang menjadi dasar kenapa tempat ini dianggap viral/direkomendasikan, atau null jika tidak ada dasar eksplisit",
          "kelurahan_desa": "nama kelurahan/desa, atau null jika tidak diketahui",
          "kecamatan": "nama kecamatan, atau null jika tidak diketahui",
          "kota_kabupaten": "nama kota/kabupaten, atau null jika tidak diketahui",
          "provinsi": "nama provinsi, atau null jika tidak diketahui",
          "negara": "nama negara, default 'Indonesia' kecuali artikel jelas menyebut negara lain",
          "latitude": angka desimal lintang jika kamu bisa simpulkan/cari dari nama tempat & lokasi, atau null jika tidak yakin,
          "longitude": angka desimal bujur jika kamu bisa simpulkan/cari dari nama tempat & lokasi, atau null jika tidak yakin,
          "presisi_koordinat": "exact" (alamat persis diketahui), "approximate" (hanya area/kecamatan/kota), atau "unknown" (tidak ada dasar koordinat),
          "catatan_lokasi": "catatan tambahan soal lokasi, mis. ketidakpastian, patokan terdekat, atau null jika tidak ada"
        }
      ]
    }
  ]
}

Jika sebuah artikel tidak menyebut tempat spesifik, kembalikan places: [] untuk artikel itu. Sertakan SEMUA article_id yang diberikan di bawah, jangan ada yang terlewat.
"""

article_ids = list(df_news_all.index)
batches = [article_ids[i:i + MANUAL_BATCH_SIZE] for i in range(0, len(article_ids), MANUAL_BATCH_SIZE)]

for b_num, batch_ids in enumerate(batches, start=1):
    parts = [PROMPT_HEADER, "\nArtikel:\n"]
    for aid in batch_ids:
        row = df_news_all.loc[aid]
        text = str(row.get("article_text", ""))[:MANUAL_MAX_CHARS]
        parts.append(f"\n### Artikel {aid}\nJudul: {row.get('title', '')}\nIsi:\n{text}\n")
    prompt_text = "\n".join(parts)
    fname = os.path.join(MANUAL_BATCH_DIR, f"batch_{b_num:03d}_prompt.txt")
    with open(fname, "w", encoding="utf-8") as f:
        f.write(prompt_text)

print(f"Dibuat {len(batches)} file prompt di folder '{MANUAL_BATCH_DIR}/'")

Dibuat 605 file prompt di folder 'manual_llm_batches_coba/'


**Setelah semua file `batch_XXX_response.json` tersimpan** di folder `manual_llm_batches_coba/` (isi tiap file harus JSON valid sesuai format di atas), jalankan sel berikut untuk menggabungkan semuanya jadi tabel final dengan kolom sesuai skema yang diminta.

In [ ]:
with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), encoding="utf-8") as f:
    id_map = json.load(f)

response_files = sorted(glob.glob(os.path.join(MANUAL_BATCH_DIR, "batch_*_response.json")))
print("File respons ditemukan:", len(response_files))

rows = []
for rf in response_files:
    with open(rf, encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError as e:
            print("Gagal parse", rf, "-> pastikan isinya JSON valid tanpa teks tambahan:", e)
            continue
    for result in data.get("results", []):
        aid = str(result.get("article_id"))
        meta = id_map.get(aid)
        if not meta:
            print("article_id tidak dikenal:", aid, "di", rf)
            continue
        for p in result.get("places", []):
            rows.append({
                "nama_tempat": p.get("nama_tempat"),
                "kategori": p.get("kategori"),
                "jenis_tempat": p.get("jenis_tempat"),
                "lokasi": p.get("lokasi"),
                "keterangan": p.get("keterangan"),
                "status_viral": p.get("status_viral"),
                "viral_sejak": p.get("viral_sejak"),
                "dasar_viral": p.get("dasar_viral"),
                "kelurahan_desa": p.get("kelurahan_desa"),
                "kecamatan": p.get("kecamatan"),
                "kota_kabupaten": p.get("kota_kabupaten"),
                "provinsi": p.get("provinsi"),
                "negara": p.get("negara"),
                "latitude": p.get("latitude"),
                "longitude": p.get("longitude"),
                "presisi_koordinat": p.get("presisi_koordinat"),
                "catatan_lokasi": p.get("catatan_lokasi"),
                "_article_key": aid,
                "keyword_pencarian": meta.get("keyword"),
                "judul_artikel_sumber": meta.get("title"),
                "link_sumber": meta.get("link") or meta.get("final_url"),
            })

df_places_raw = pd.DataFrame(rows).dropna(subset=["nama_tempat"]).reset_index(drop=True)
print("Total baris tempat mentah (sebelum dedup antar-artikel):", len(df_places_raw))
df_places_raw.head(5)

Tempat yang sama bisa disebut di beberapa artikel berbeda (dengan penulisan nama sedikit berbeda). Sel berikut melakukan **fuzzy dedup** nama tempat (`rapidfuzz.fuzz.token_sort_ratio`) supaya tiap tempat jadi satu baris, lalu menghitung `jumlah_artikel` dan menyimpan ke CSV final dengan urutan kolom sesuai skema.

In [ ]:
FUZZY_THRESHOLD = 88  # 0-100, makin tinggi makin ketat kemiripan nama

FINAL_COLUMNS = [
    "no", "nama_tempat", "kategori", "jenis_tempat", "lokasi", "keterangan",
    "status_viral", "viral_sejak", "dasar_viral", "kelurahan_desa", "kecamatan",
    "kota_kabupaten", "provinsi", "negara", "latitude", "longitude",
    "presisi_koordinat", "catatan_lokasi", "jumlah_artikel",
    "keyword_pencarian", "judul_artikel_sumber", "link_sumber",
]


def normalize_place_name(name: str) -> str:
    name = str(name).lower().strip()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def first_non_null(series: pd.Series):
    valid = series.dropna()
    return valid.iloc[0] if len(valid) else None


df_places_raw["_nama_norm"] = df_places_raw["nama_tempat"].map(normalize_place_name)

# cluster nama unik yang mirip (greedy): tiap nama unik dibandingkan ke representative cluster yang sudah ada
unique_names = df_places_raw["_nama_norm"].unique().tolist()
name_to_group = {}
cluster_reps = []  # representative nama tiap cluster, indexnya = id cluster

for name in unique_names:
    if not name:
        continue
    best_idx, best_score = None, 0
    for idx, rep in enumerate(cluster_reps):
        score = fuzz.token_sort_ratio(name, rep)
        if score > best_score:
            best_idx, best_score = idx, score
    if best_score >= FUZZY_THRESHOLD:
        name_to_group[name] = best_idx
    else:
        cluster_reps.append(name)
        name_to_group[name] = len(cluster_reps) - 1

df_places_raw["canonical_group"] = df_places_raw["_nama_norm"].map(name_to_group)
print("Total nama mentah unik:", len(unique_names))
print("Total tempat unik setelah fuzzy dedup:", df_places_raw["canonical_group"].nunique())

agg_cols = [c for c in FINAL_COLUMNS if c not in ("no", "jumlah_artikel")]
grouped_rows = []
for _, g in df_places_raw.groupby("canonical_group", sort=False):
    row = {col: first_non_null(g[col]) for col in agg_cols if col in g.columns}
    row["jumlah_artikel"] = g["_article_key"].nunique()
    grouped_rows.append(row)

df_places_final = pd.DataFrame(grouped_rows, columns=[c for c in FINAL_COLUMNS if c != "no"])
df_places_final.insert(0, "no", range(1, len(df_places_final) + 1))

OUTPUT_CSV = "hasil_ekstraksi_tempat_viral.csv"
df_places_final.to_csv(OUTPUT_CSV, index=False)
print(f"Total baris tempat (setelah dedup): {len(df_places_final)}. Disimpan ke {OUTPUT_CSV}")
df_places_final.head(5)